# Using discopt as a Pyomo solver

discopt ships an optional [Pyomo](https://pyomo.org) {cite:p}`Hart2017` plugin so that models written with Pyomo's algebraic modeling layer can be solved with discopt through the familiar `SolverFactory` interface — no rewrite required.

Install the extra:

```bash
pip install discopt[pyomo]
```

Under the hood the bridge round-trips the model through a temporary AMPL `.nl` file {cite:p}`Fourer2003`: Pyomo's `.nl` writer and discopt's `from_nl` agree on the variable *column order*, so the solution maps back by index. This notebook shows a basic solve, solver options, and dual values.

## Activate the plugin

Importing `discopt.pyomo` registers the `'discopt'` solver with Pyomo's `SolverFactory` (a `pyomo.solvers` entry point does the same automatically after installation).

In [1]:
import pyomo.environ as pyo
import discopt.pyomo  # registers the 'discopt' solver

print("discopt registered:", "discopt" in pyo.SolverFactory)
opt = pyo.SolverFactory("discopt")
print("available:", opt.available())

discopt registered: True
available: True


## A small MINLP

A mixed-integer nonlinear program with one continuous variable, one binary variable, a quadratic objective, and a linear constraint:

$$\min_{x,\,y}\;(x-3)^2 + 2y \quad\text{s.t.}\quad x + 5y \ge 4,\; x \in [0,10],\; y \in \{0,1\}.$$

The optimum is $x=4,\,y=0$ with objective $1$.

In [2]:
m = pyo.ConcreteModel()
m.x = pyo.Var(bounds=(0, 10))
m.y = pyo.Var(domain=pyo.Binary)
m.obj = pyo.Objective(expr=(m.x - 3) ** 2 + 2 * m.y, sense=pyo.minimize)
m.c = pyo.Constraint(expr=m.x + 5 * m.y >= 4)

results = opt.solve(m)
print("termination:", results.solver.termination_condition)
print("objective:  ", pyo.value(m.obj))
print("x =", m.x.value, "  y =", m.y.value)

termination: optimal
objective:   0.9999999225074985
x = 3.9999999612537485   y = 0.0


discopt solved the model and loaded the solution back onto the Pyomo variables. The binary variable comes back as an exact integer (integrality drift is rounded away on load).

## Solver options

Standard Pyomo options map onto discopt's `solve()` keywords: `timelimit` → `time_limit`, `mipgap` → `gap_tolerance`, `threads` → `threads`, and `tee=True` streams progress. Unrecognised options are forwarded verbatim, so any discopt solve keyword works without a plugin change.

In [3]:
results = opt.solve(m, options={"timelimit": 30, "mipgap": 1e-4})
print("termination:", results.solver.termination_condition)
print("wall time (s):", results.solver.wallclock_time)

termination: optimal
wall time (s): 0.004830209072679281


## Dual values and reduced costs

Declare an `import` `Suffix` and discopt's KKT multipliers are loaded best-effort. Consider a convex NLP whose optimum sits on an active constraint:

$$\min_x\;(x-3)^2 \quad\text{s.t.}\quad x \ge 4.$$

The optimum is $x=4$, and the constraint multiplier is $2$ (the AMPL/Pyomo sign convention, matching e.g. Ipopt).

In [4]:
n = pyo.ConcreteModel()
n.x = pyo.Var(bounds=(0, 10))
n.obj = pyo.Objective(expr=(n.x - 3) ** 2, sense=pyo.minimize)
n.c = pyo.Constraint(expr=n.x >= 4)
n.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
n.rc = pyo.Suffix(direction=pyo.Suffix.IMPORT)

opt.solve(n)
print("x =", round(n.x.value, 6))
print("dual[c] =", n.dual.get(n.c))
print("rc[x]   =", n.rc.get(n.x))

x = 4.0
dual[c] = 1.999999922298516
rc[x]   = 2.0898053236939798e-10


Duals are populated only when discopt exposes multipliers (the NLP/KKT path); for problems solved by another route the `Suffix` is simply left empty — values are never fabricated.

## How it works and limitations

Pyomo's `.nl` writer emits the model in its original variable space (presolve and scaling disabled), discopt's `from_nl` reads it with the **same column/row order**, and the solution maps back by index. As a result:

- Variable and constraint *values* map by column/row order, not by name (Pyomo and discopt name objects differently).
- Constraint names are not preserved through `.nl` (cosmetic; solving and duals are unaffected).
- A model whose operators discopt's reader does not support returns a structured `error` result rather than crashing.

See [docs/pyomo_solver.md](../pyomo_solver.md) for the full reference.